#Initialization

In [0]:
import pyspark.sql.functions as F

#Read Bronze Table

In [0]:
patients = spark.table("health.bronze.patients")

#Data Understanding

###Review Schema

In [0]:
patients.printSchema()

###Review Sample Records

In [0]:
display(
    patients.limit(10)
)

#Data Profiling

###Check Null Values

In [0]:
display(
    patients.select(
        [
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(c)
            for c in patients.columns
        ]
    )
)


###Whitespace Validation

In [0]:
string_cols = [
    field.name
    for field in patients.schema.fields
    if field.dataType.simpleString() == "string"
]

display(
    patients.select(
        [
            F.sum(
                F.when(F.col(c) != F.trim(F.col(c)), 1).otherwise(0)
            ).alias(c)
            for c in string_cols
        ]
    )
)

###Business Key Validation

In [0]:
total_rows = patients.count()

distinct_patient_ids = (
    patients
    .select("patient_id")
    .distinct()
    .count()
)

print(f"Total Rows: {total_rows}")
print(f"Distinct Patient IDs: {distinct_patient_ids}")

###Duplicate Analysis

In [0]:
duplicate_patients = (
    patients
    .groupBy("patient_id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_patients)

###Review Duplicate Records

In [0]:
duplicate_ids = (
    duplicate_patients
    .select("patient_id")
)

display(
    patients.join(
        duplicate_ids,
        on="patient_id",
        how="inner"
    )
    .orderBy("patient_id")
)

###Gender Validation

In [0]:
display(
    patients.groupBy("gender")
    .count()
    .orderBy("gender")
)

###Exact Duplicate Validation

In [0]:
print(f"Total Rows: {patients.count()}")

print(
    f"Rows After dropDuplicates(): {patients.dropDuplicates().count()}"
)

#Transformations

###Remove Duplicate Patient Records

In [0]:
patients_clean = patients.dropDuplicates()

print(f"Rows Before: {patients.count()}")
print(f"Rows After: {patients_clean.count()}")

###Handle Missing City Values

In [0]:
valid_patients = patients_clean.filter(
    F.col("city").isNotNull()
)

patients_missing_city = patients_clean.filter(
    F.col("city").isNull()
)

###Check Results

In [0]:
print(f"Valid Patients: {valid_patients.count()}")
print(f"Patients Missing City: {patients_missing_city.count()}")
print(f"Total Patients: {patients_clean.count()}")

#Write Silver Tables

###Save Valid Patients

In [0]:
(
    valid_patients.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("health.silver.patients")
)

##Save Patients Missing City

In [0]:
(
    patients_missing_city.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("health.silver.patients_missing_city")
)

###Verify Silver Patients

In [0]:
display(
    spark.table("health.silver.patients")
    .limit(10)
)

###Verify Patients Missing City

In [0]:
display(
    spark.table("health.silver.patients_missing_city")
    .limit(10)
)